# Chapter 13 figure — LLM Limitations & Evaluation

**`gen_reliability_diagram.py` → notebook.** Reproduces a figure from *Large Language Models in Finance*. Run the cells top to bottom. In Google Colab, run the install cell first; locally you can skip it if your environment is already set up (`pip install -r requirements.txt`).

The figure is both saved to disk *and* shown inline below.

In [ ]:
# Colab: run once to install this figure's dependencies.
# (Locally, skip if you already ran `pip install -r requirements.txt`.)
%pip install matplotlib numpy

## What this figure shows

```
Deterministic generator for the ch13 reliability diagram (fig:ch13-reliability).

Uses the fixed bin statistics from Example ex:ch13-ece-credit (no data/network).
Bins of width 0.2; mean predicted probability vs empirical accuracy per bin.
The gap between the bars and the diagonal is the per-bin calibration error that ECE
aggregates (ECE = 0.115 in the example).

Run:  python3 gen_reliability_diagram.py
Output: ../../book/chapters/13-llm-limitations-evaluation/figures/fig_reliability.pdf
```

In [ ]:
# --- notebook shim: let the script run unchanged outside its repo ---
import sys, pathlib
sys.argv = ['gen_reliability_diagram.py']  # use the script's argparse defaults
# The script derives its save path from __file__ (3 levels up = repo root).
# Point it at a writable ./outputs tree so the save works anywhere
# (Colab, local Jupyter). The figure is also shown inline below.
__file__ = str(pathlib.Path.cwd() / 'outputs' / 'code' / 'figures' / '13-llm-limitations-evaluation' / 'gen_reliability_diagram.py')

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Fixed bins from ex:ch13-ece-credit.
CONF = np.array([0.08, 0.31, 0.50, 0.71, 0.88])   # mean predicted prob per bin
ACC = np.array([0.11, 0.28, 0.47, 0.55, 0.62])    # empirical accuracy per bin
COUNT = np.array([120, 180, 200, 300, 200])

OUT = (Path(__file__).resolve().parents[3]
       / "book/chapters/13-llm-limitations-evaluation/figures/fig_reliability.pdf")


def main():
    ece = float(np.sum(COUNT / COUNT.sum() * np.abs(CONF - ACC)))

    fig, ax = plt.subplots(figsize=(5.6, 5.4))
    ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
    ax.bar(CONF, ACC, width=0.16, color="#4c72b0", edgecolor="black",
           linewidth=0.5, alpha=0.85, label="model accuracy")
    # Mark the calibration gap for each bin.
    for c, a in zip(CONF, ACC):
        ax.plot([c, c], [a, c], color="#d62728", linewidth=1.2)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean predicted probability (confidence)")
    ax.set_ylabel("Empirical accuracy")
    ax.set_title(f"Reliability diagram (ECE = {ece:.3f})")
    ax.legend(loc="upper left")
    ax.set_aspect("equal")
    fig.tight_layout()
    OUT.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT, bbox_inches="tight")
    print(f"wrote {OUT} (ECE={ece:.3f})")

# render the figure inline (in addition to the file the script saves)
main()
import matplotlib.pyplot as _plt; _plt.show()